In [1]:
# Cellule 1 : imports, graine, chemins
import os, glob, json, random
import numpy as np, pandas as pd
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("device :", dev)

hits = glob.glob("/kaggle/input/**/train.tsv", recursive=True)
assert hits, "train.tsv introuvable : ajoute ton dataset avec Add Input"
P = os.path.dirname(hits[0]) + "/"

def has_jpg(d):
    return os.path.isdir(d) and any(f.endswith(".jpg") for f in os.listdir(d)[:20])

IMG = next((d for d in (P + "images/images/", P + "images/") if has_jpg(d)), None)
assert IMG, "dossier d'images introuvable dans " + P

OUT = "/kaggle/working/"
os.makedirs(OUT + "emb", exist_ok=True)
os.makedirs(OUT + "results", exist_ok=True)
print("P   =", P)
print("IMG =", IMG)

device : cuda
P   = /kaggle/input/datasets/sahnounsahnoun/data-fakenewes/
IMG = /kaggle/input/datasets/sahnounsahnoun/data-fakenewes/images/images/


In [2]:
# Cellule 2 : chargement des splits et controles d'integrite
S = ["train", "val", "test"]
df = {s: pd.read_csv(P + s + ".tsv", sep="\t") for s in S}

# version NETTOYEE du dataset (celle de E1, E2, E3 : test = 1 396 publications)
VERIFIER_TAILLES = True
ATTENDU = {"train": 7432, "val": 1403, "test": 1396}
if VERIFIER_TAILLES:
    for s in S:
        assert len(df[s]) == ATTENDU[s], (
            "%s : %d publications, %d attendues. Ce n'est pas la version nettoyee du dataset "
            "(l'ancienne version non nettoyee a 7792 / 1469 / 1450). Retire l'ancien dataset des Inputs "
            "et ajoute la version nettoyee." % (s, len(df[s]), ATTENDU[s]))

have = {f[:-4] for f in os.listdir(IMG)}
for s in S:
    miss = int((~df[s]["id"].astype(str).isin(have)).sum())
    print(s, len(df[s]), "exemples | images manquantes :", miss)
    assert miss == 0, "images manquantes : verifie la version du dataset"

y = {s: df[s]["2_way_label"].values for s in S}
y6 = {s: df[s]["6_way_label"].values for s in S}

# les probabilites d'E1 doivent correspondre ligne par ligne aux splits
e1 = {s: np.load(P + "e1_" + s + "_proba.npy") for s in ["val", "test"]}
for s in ["val", "test"]:
    assert len(e1[s]) == len(df[s]), (
        "e1_%s_proba.npy a %d lignes, %s.tsv en a %d : mets a jour le dataset "
        "avec les .npy regeneres apres le nettoyage" % (s, len(e1[s]), s, len(df[s])))
print("alignement E1 / splits : OK")

train 7432 exemples | images manquantes : 0
val 1403 exemples | images manquantes : 0
test 1396 exemples | images manquantes : 0
alignement E1 / splits : OK


In [3]:
# Cellule 3 : embeddings image, ResNet-18 gele (512 dimensions)
tf = transforms.Compose([transforms.ToTensor(),
                         transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

class Imgs(Dataset):
    def __init__(self, ids):
        self.ids = list(ids)
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        return tf(Image.open(IMG + self.ids[i] + ".jpg").convert("RGB"))

resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet.fc = torch.nn.Identity()
resnet = resnet.to(dev).eval()

def img_emb(ids):
    out = []
    for b in DataLoader(Imgs(ids), batch_size=64, shuffle=False, num_workers=2):
        with torch.no_grad():
            out.append(resnet(b.to(dev)).cpu().numpy())
    return np.concatenate(out).astype("float32")

Xi = {}
for s in S:
    Xi[s] = img_emb(df[s]["id"].astype(str))
    np.save(OUT + "emb/img_" + s + ".npy", Xi[s])
    print("image", s, Xi[s].shape)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 190MB/s]


image train (7432, 512)
image val (1403, 512)
image test (1396, 512)


In [4]:
# Cellule 4 : embeddings texte, DistilBERT gele (768 dimensions, moyenne des tokens)
from transformers import AutoTokenizer, AutoModel

tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert = AutoModel.from_pretrained("distilbert-base-uncased").to(dev).eval()

def txt_emb(texts):
    texts = list(texts)
    out = []
    for i in range(0, len(texts), 64):
        t = tok(texts[i:i + 64], padding=True, truncation=True, max_length=64, return_tensors="pt").to(dev)
        with torch.no_grad():
            h = bert(**t).last_hidden_state
        m = t["attention_mask"].unsqueeze(-1)
        out.append(((h * m).sum(1) / m.sum(1)).cpu().numpy())
    return np.concatenate(out).astype("float32")

Xt = {}
for s in S:
    Xt[s] = txt_emb(df[s]["clean_title"].astype(str))
    np.save(OUT + "emb/txt_" + s + ".npy", Xt[s])
    print("texte", s, Xt[s].shape)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


texte train (7432, 768)
texte val (1403, 768)
texte test (1396, 768)


In [5]:
# Cellule 5 : fonctions d'evaluation et d'entrainement (regression logistique sur embeddings)
GRID = [0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1]

def metrics(y_true, p):
    pred = (p > 0.5).astype(int)
    return dict(acc=accuracy_score(y_true, pred),
                prec=precision_score(y_true, pred, average="macro", zero_division=0),
                rec=recall_score(y_true, pred, average="macro", zero_division=0),
                macro_f1=f1_score(y_true, pred, average="macro", zero_division=0),
                auc=roc_auc_score(y_true, p))

def run_lr(name, X, grid=None):
    grid = grid or GRID   # par defaut : la grille d'origine, donc E2 et E3 ne changent pas
    mk = lambda C: make_pipeline(StandardScaler(), LogisticRegression(
        C=C, max_iter=3000, class_weight="balanced", random_state=SEED))
    # choix de C sur la validation uniquement
    val = {C: f1_score(y["val"], mk(C).fit(X["train"], y["train"]).predict(X["val"]), average="macro")
           for C in grid}
    best = max(val, key=val.get)
    print(name, "Macro F1 validation par C :", {c: round(v, 4) for c, v in val.items()})
    if best in (grid[0], grid[-1]):
        print("attention : le meilleur C est en bord de grille")
    clf = mk(best).fit(X["train"], y["train"])
    pv = clf.predict_proba(X["val"])[:, 1]
    pt = clf.predict_proba(X["test"])[:, 1]
    m = metrics(y["test"], pt)
    print(name, "meilleur C =", best, "| val F1 =", round(val[best], 4),
          "| TEST", {k: round(float(v), 4) for k, v in m.items()})
    np.save(OUT + "results/" + name.lower() + "_val_proba.npy", pv)
    np.save(OUT + "results/" + name.lower() + "_test_proba.npy", pt)
    return dict(best_C=best, val_macro_f1=val[best], test=m), pt

In [6]:
# Cellule 6 : E3 (image seule) et E2 (texte seul, DistilBERT gele)
res, proba = {}, {}
res["E3"], proba["E3"] = run_lr("E3", Xi)
res["E2"], proba["E2"] = run_lr("E2", Xt)
proba["E1"] = e1["test"]   # probabilites d'E1 (TF-IDF), deja calculees sur le PC

E3 Macro F1 validation par C : {0.0003: 0.6711, 0.001: 0.6804, 0.003: 0.682, 0.01: 0.6799, 0.03: 0.6743, 0.1: 0.6693, 0.3: 0.6657, 1: 0.665}
E3 meilleur C = 0.003 | val F1 = 0.682 | TEST {'acc': 0.6991, 'prec': 0.7001, 'rec': 0.6989, 'macro_f1': 0.6986, 'auc': 0.7547}
E2 Macro F1 validation par C : {0.0003: 0.796, 0.001: 0.8018, 0.003: 0.8068, 0.01: 0.8068, 0.03: 0.8082, 0.1: 0.8104, 0.3: 0.8075, 1: 0.8096}
E2 meilleur C = 0.1 | val F1 = 0.8104 | TEST {'acc': 0.8009, 'prec': 0.8009, 'rec': 0.8008, 'macro_f1': 0.8008, 'auc': 0.8615}


In [7]:
# Cellule 7 : tableau comparatif sur le test, et precision par type de faux
tab = pd.DataFrame({
    "Majoritaire (tout vrai)": metrics(y["test"], np.ones(len(y["test"]))),
    "E1 texte TF-IDF": metrics(y["test"], proba["E1"]),
    "E2 texte DistilBERT": res["E2"]["test"],
    "E3 image ResNet-18": res["E3"]["test"],
}).T[["acc", "macro_f1", "auc"]].astype(float).round(4)
print(tab, "\n")

names = {0: "vrai", 1: "satire", 2: "trompeur", 3: "bots", 4: "image manipulee", 5: "fausse connexion"}
per_type = pd.DataFrame({"n": pd.Series(y6["test"]).value_counts().sort_index()})
for k in ["E1", "E2", "E3"]:
    ok = (proba[k] > 0.5).astype(int) == y["test"]
    per_type[k] = pd.Series(ok).groupby(y6["test"]).mean().round(3)
per_type.index = [str(i) + " " + names.get(i, "?") for i in per_type.index]
print("Precision par type (test) :")
print(per_type)

# diagnostic du raccourci : predire avec le seul subreddit (ce n'est PAS un modele a comparer)
maj = df["train"].groupby("subreddit")["2_way_label"].agg(lambda s: s.mode()[0])
sp = df["test"]["subreddit"].map(maj)
print("\nlignes de test dont le subreddit est absent du train :", int(sp.isna().sum()))
sp = sp.fillna(1).astype(int)
sub_diag = dict(acc=accuracy_score(y["test"], sp), macro_f1=f1_score(y["test"], sp, average="macro"))
print("Diagnostic : le subreddit seul donne", {k: round(v, 4) for k, v in sub_diag.items()})

                            acc  macro_f1     auc
Majoritaire (tout vrai)  0.5029    0.3346  0.5000
E1 texte TF-IDF          0.7507    0.7502  0.8218
E2 texte DistilBERT      0.8009    0.8008  0.8615
E3 image ResNet-18       0.6991    0.6986  0.7547 

Precision par type (test) :
                      n     E1     E2     E3
0 vrai              702  0.791  0.803  0.736
1 satire             63  0.540  0.603  0.587
2 trompeur          212  0.618  0.675  0.486
3 bots               21  0.238  0.476  0.333
4 image manipulee   347  0.833  0.942  0.778
5 fausse connexion   51  0.667  0.706  0.824

lignes de test dont le subreddit est absent du train : 0
Diagnostic : le subreddit seul donne {'acc': 1.0, 'macro_f1': 1.0}


In [8]:
# Cellule 8 : sauvegarde des metriques et liste des fichiers produits
log = dict(seed=SEED, device=dev,
           n={s: len(df[s]) for s in S},
           E1=metrics(y["test"], proba["E1"]), E2=res["E2"], E3=res["E3"],
           table=tab.to_dict(), per_type=per_type.to_dict(), subreddit_diagnostic=sub_diag)
with open(OUT + "results/e2_e3_metrics.json", "w", encoding="utf-8") as f:
    json.dump(log, f, indent=2, ensure_ascii=False,
              default=lambda o: o.item() if hasattr(o, "item") else str(o))

for root, _, files in os.walk(OUT):
    for fn in sorted(files):
        p = os.path.join(root, fn)
        print(p, round(os.path.getsize(p) / 1e6, 2), "Mo")

/kaggle/working/__notebook__.ipynb 0.04 Mo
/kaggle/working/results/e2_e3_metrics.json 0.0 Mo
/kaggle/working/results/e2_test_proba.npy 0.01 Mo
/kaggle/working/results/e2_val_proba.npy 0.01 Mo
/kaggle/working/results/e3_test_proba.npy 0.01 Mo
/kaggle/working/results/e3_val_proba.npy 0.01 Mo
/kaggle/working/emb/img_test.npy 2.86 Mo
/kaggle/working/emb/img_train.npy 15.22 Mo
/kaggle/working/emb/img_val.npy 2.87 Mo
/kaggle/working/emb/txt_test.npy 4.29 Mo
/kaggle/working/emb/txt_train.npy 22.83 Mo
/kaggle/working/emb/txt_val.npy 4.31 Mo


## Partie 2 : E3 avec d'autres réseaux image

**Question :** le résultat de E3 (0,699 avec ResNet-18) dépend-il du réseau image choisi ? Seul le réseau change : mêmes images 224×224, même régression logistique, mêmes splits.

| Étape | Réseau | Appris sur | Vecteur par image |
|---|---|---|---|
| E3ctrl | ResNet-18 (contrôle, embeddings de la cellule 3) | ImageNet | 512 |
| E3b | EfficientNet-B0 | ImageNet | 1280 |
| E3c | ViT-B/16 | ImageNet | 768 |
| E3d | CLIP ViT-B/32 | 400 millions de paires image-texte du web | 512 |

**Pourquoi CLIP :** il a appris sur des images du web accompagnées de texte, ce qui ressemble davantage aux publications Reddit (mèmes, captures d'écran, photos) que les objets d'ImageNet. C'est une hypothèse à tester, pas un résultat.

**Grille de C étendue** jusqu'à 0,00001 pour ces quatre réseaux : le meilleur C d'EfficientNet-B0 et de ViT-B/16 était au bord de l'ancienne grille. E2 et E3 (partie 1) gardent la grille d'origine, donc leurs chiffres ne bougent pas. Le contrôle E3ctrl est le même ResNet-18 sur la grille étendue : on compare les autres réseaux à E3ctrl.

**Chaque réseau est comparé à ResNet-18 avec un intervalle à 95 %** (bootstrap sur les 1 396 publications du test). Si l'intervalle contient 0, l'écart n'est pas établi. Le réseau retenu se choisit avec la **validation**.

In [9]:
# Cellule 9 : embeddings image avec d'autres reseaux geles (meme pretraitement que E3)
import unicodedata
from tqdm.auto import tqdm
from transformers import CLIPVisionModelWithProjection

NUM_WORKERS = 2   # mets 0 si le calcul semble bloque
IMAGENET = ([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
CLIP_STATS = ([0.4815, 0.4578, 0.4082], [0.2686, 0.2613, 0.2758])

# ViT-B/16 et CLIP exigent des images de 224 x 224 : on verifie
taille = Image.open(IMG + str(df["train"]["id"].iloc[0]) + ".jpg").size
assert taille == (224, 224), "les images doivent etre en 224x224, trouve : %s" % (taille,)

class ImgsTf(Dataset):
    def __init__(self, ids, tfm):
        self.ids, self.tfm = list(ids), tfm
    def __len__(self):
        return len(self.ids)
    def __getitem__(self, i):
        return self.tfm(Image.open(IMG + self.ids[i] + ".jpg").convert("RGB"))

def torchvision_net(ctor, poids, tete):
    def build():
        net = ctor(weights=poids)                 # reseau pre-entraine
        setattr(net, tete, torch.nn.Identity())   # on garde le vecteur, pas les 1000 classes
        return net
    return build

class ClipImg(torch.nn.Module):
    """Encodeur image de CLIP ViT-B/32 : vecteur de 512 nombres par image."""
    def __init__(self):
        super().__init__()
        self.m = CLIPVisionModelWithProjection.from_pretrained("openai/clip-vit-base-patch32")
    def forward(self, x):
        return self.m(pixel_values=x).image_embeds

# nom -> (constructeur, (moyenne, ecart-type) de normalisation)
RESEAUX = {
    "efficientnet_b0": (torchvision_net(models.efficientnet_b0, models.EfficientNet_B0_Weights.IMAGENET1K_V1, "classifier"), IMAGENET),
    "vit_b_16": (torchvision_net(models.vit_b_16, models.ViT_B_16_Weights.IMAGENET1K_V1, "heads"), IMAGENET),
    "clip_vit_b32": (ClipImg, CLIP_STATS),
}
ETAPES = {"resnet18": ("E3ctrl", "ResNet-18"), "efficientnet_b0": ("E3b", "EfficientNet-B0"),
          "vit_b_16": ("E3c", "ViT-B/16"), "clip_vit_b32": ("E3d", "CLIP ViT-B/32")}

def extraire(nom):
    build, (moy, ety) = RESEAUX[nom]
    tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize(moy, ety)])
    net = build().to(dev).eval()
    X = {}
    for s in S:
        out = []
        lots = DataLoader(ImgsTf(df[s]["id"].astype(str), tfm), batch_size=64, shuffle=False, num_workers=NUM_WORKERS)
        for b in tqdm(lots, desc="%s %s" % (nom, s)):
            with torch.no_grad():
                out.append(net(b.to(dev)).cpu().numpy())
        X[s] = np.concatenate(out).astype("float32")
        np.save(OUT + "emb/img_%s_%s.npy" % (nom, s), X[s])
        print(nom, s, X[s].shape)
    del net
    torch.cuda.empty_cache()
    return X

Xe = {"resnet18": Xi}     # ResNet-18 : embeddings de la cellule 3 (ils servent de controle)
for nom in RESEAUX:
    Xe[nom] = extraire(nom)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 146MB/s]


efficientnet_b0 train:   0%|          | 0/117 [00:00<?, ?it/s]

efficientnet_b0 train (7432, 1280)


efficientnet_b0 val:   0%|          | 0/22 [00:00<?, ?it/s]

efficientnet_b0 val (1403, 1280)


... (18 outputs left out)

clip_vit_b32 val:   0%|          | 0/22 [00:00<?, ?it/s]

clip_vit_b32 val (1403, 512)


clip_vit_b32 test:   0%|          | 0/22 [00:00<?, ?it/s]

clip_vit_b32 test (1396, 512)


In [10]:
# Cellule 10 : regression logistique pour les quatre reseaux (grille etendue), ecart face a ResNet-18
GRID_ETENDUE = [0.00001, 0.00003, 0.0001, 0.0003, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3, 1]
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20)   # tableaux complets

def gain_points(y_true, p_a, p_b, B=1000):
    """Ecart de Macro F1 (a - b) en points, avec intervalle a 95 % par bootstrap sur le test."""
    rng = np.random.default_rng(SEED)
    pa, pb = (p_a > 0.5).astype(int), (p_b > 0.5).astype(int)
    g = 100 * (f1_score(y_true, pa, average="macro") - f1_score(y_true, pb, average="macro"))
    n, d = len(y_true), []
    for _ in range(B):
        i = rng.integers(0, n, n)
        d.append(100 * (f1_score(y_true[i], pa[i], average="macro") - f1_score(y_true[i], pb[i], average="macro")))
    return g, tuple(np.percentile(d, [2.5, 97.5]))

res2, proba2 = {}, {}
for nom in Xe:
    etape = ETAPES[nom][0]
    res2[nom], proba2[nom] = run_lr(etape, Xe[nom], grid=GRID_ETENDUE)
    pv = np.load(OUT + "results/" + etape.lower() + "_val_proba.npy")
    res2[nom]["val"] = metrics(y["val"], pv)

n_test = len(y["test"])
lignes = {"E3 officiel ResNet-18 (grille d'origine)": dict(
    val_macro_f1=res["E3"]["val_macro_f1"], acc=res["E3"]["test"]["acc"], macro_f1=res["E3"]["test"]["macro_f1"],
    auc=res["E3"]["test"]["auc"], marge_acc_95=1.96 * np.sqrt(res["E3"]["test"]["acc"] * (1 - res["E3"]["test"]["acc"]) / n_test))}
for nom in Xe:
    m = res2[nom]["test"]
    etape, lib = ETAPES[nom]
    lignes["%s %s" % (etape, lib)] = dict(
        val_macro_f1=res2[nom]["val_macro_f1"], acc=m["acc"], macro_f1=m["macro_f1"], auc=m["auc"],
        marge_acc_95=1.96 * np.sqrt(m["acc"] * (1 - m["acc"]) / n_test))
tab2 = pd.DataFrame(lignes).T.astype(float).round(4)
print("\nTest (%d exemples) ; marge_acc_95 = demi-largeur de l'intervalle a 95 %% de l'exactitude :" % n_test)
print(tab2, "\n")

ecart_grille = res2["resnet18"]["test"]["macro_f1"] - res["E3"]["test"]["macro_f1"]
print("Controle : E3ctrl (grille etendue) - E3 officiel (grille d'origine) = %+.4f de Macro F1" % ecart_grille)
if abs(ecart_grille) > 0.005:
    print("  La grille etendue change un peu le ResNet-18 : compare les autres reseaux a E3ctrl, pas a E3.")

gains = {}
print("\nEcart de Macro F1 face a ResNet-18 / E3ctrl (points, intervalle a 95 %, bootstrap sur le test) :")
for nom in Xe:
    if nom == "resnet18":
        continue
    g, (lo, hi) = gain_points(y["test"], proba2[nom], proba2["resnet18"])
    gains[nom] = (g, lo, hi)
    verdict = "ecart non etabli" if lo <= 0 <= hi else ("meilleur que ResNet-18" if lo > 0 else "moins bon que ResNet-18")
    print("  %-18s %+5.1f  [%+5.1f ; %+5.1f]  -> %s" % (ETAPES[nom][1], g, lo, hi, verdict))

# correspondance des types a 6 classes : a verifier (diagnostic en cours), memes noms que la cellule 7
per_type2 = pd.DataFrame({"n": pd.Series(y6["test"]).value_counts().sort_index()})
for nom in Xe:
    ok = (proba2[nom] > 0.5).astype(int) == y["test"]
    per_type2[nom] = pd.Series(ok).groupby(y6["test"]).mean().round(3)
per_type2.index = [str(i) + " " + names.get(i, "?") for i in per_type2.index]
print("\nPrecision par type (test) :")
print(per_type2)

best = max(Xe, key=lambda k: res2[k]["val_macro_f1"])
print("\nReseau le meilleur sur la VALIDATION :", best, "(Macro F1 val = %.4f)" % res2[best]["val_macro_f1"])
print("Rappel : on choisit avec la validation, on rapporte le test. Une seule graine : un ecart dont "
      "l'intervalle contient 0 n'est pas une difference etablie.")

E3ctrl Macro F1 validation par C : {1e-05: 0.6547, 3e-05: 0.6589, 0.0001: 0.6696, 0.0003: 0.6711, 0.001: 0.6804, 0.003: 0.682, 0.01: 0.6799, 0.03: 0.6743, 0.1: 0.6693, 0.3: 0.6657, 1: 0.665}
E3ctrl meilleur C = 0.003 | val F1 = 0.682 | TEST {'acc': 0.6991, 'prec': 0.7001, 'rec': 0.6989, 'macro_f1': 0.6986, 'auc': 0.7547}
E3b Macro F1 validation par C : {1e-05: 0.6687, 3e-05: 0.6709, 0.0001: 0.6817, 0.0003: 0.6727, 0.001: 0.6622, 0.003: 0.6628, 0.01: 0.645, 0.03: 0.6358, 0.1: 0.6279, 0.3: 0.6265, 1: 0.6272}
E3b meilleur C = 0.0001 | val F1 = 0.6817 | TEST {'acc': 0.6719, 'prec': 0.6722, 'rec': 0.6718, 'macro_f1': 0.6717, 'auc': 0.7386}
E3c Macro F1 validation par C : {1e-05: 0.6482, 3e-05: 0.6547, 0.0001: 0.6683, 0.0003: 0.6787, 0.001: 0.6831, 0.003: 0.6796, 0.01: 0.669, 0.03: 0.672, 0.1: 0.6699, 0.3: 0.6706, 1: 0.6706}
E3c meilleur C = 0.001 | val F1 = 0.6831 | TEST {'acc': 0.6698, 'prec': 0.6698, 'rec': 0.6697, 'macro_f1': 0.6697, 'auc': 0.7428}
E3d Macro F1 validation par C : {1e-05:

In [11]:
# Cellule 11 : fichiers de resultats pour MLflow (un JSON par reseau) et metriques de la partie 2
os.makedirs(OUT + "results/mlflow", exist_ok=True)

def slug(t):
    return "_".join(unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode().lower().split())

AVERT = ("Types a 6 classes : correspondance des etiquettes en cours de verification ; "
         "fausse_connexion : %d publications ; bots : %d." % ((y6["test"] == 5).sum(), (y6["test"] == 3).sum()))
for nom in Xe:
    etape, lib = ETAPES[nom]
    m, v = res2[nom]["test"], res2[nom]["val"]
    met = dict(accuracy=m["acc"], f1_macro=m["macro_f1"], auc=m["auc"],
               val_accuracy=v["acc"], val_f1_macro=v["macro_f1"], val_auc=v["auc"])
    if nom in gains:
        met.update(ecart_f1_vs_resnet18_points=gains[nom][0], ecart_f1_ic95_bas=gains[nom][1], ecart_f1_ic95_haut=gains[nom][2])
    ok = (proba2[nom] > 0.5).astype(int) == y["test"]
    for t in sorted(np.unique(y6["test"])):
        masque = y6["test"] == t
        met["acc_type_" + slug(names.get(int(t), str(t)))] = float(ok[masque].mean())
        met["n_type_" + slug(names.get(int(t), str(t)))] = int(masque.sum())
    out = dict(
        etape=etape, nom=nom + "_gele_lr",
        params=dict(entree="image", representation=lib + " (gele)",
                    classifieur="LogisticRegression, class_weight=balanced, C choisi sur la validation (grille 1e-5 a 1)",
                    C_retenu=res2[nom]["best_C"], dim_embedding=int(Xe[nom]["train"].shape[1]),
                    n_train=len(df["train"]), n_val=len(df["val"]), n_test=len(df["test"])),
        metrics={k: float(val) for k, val in met.items()},
        tags=dict(source="notebook Kaggle 02_embeddings_e2_e3 (partie 2)", avertissement_types=AVERT))
    with open(OUT + "results/mlflow/resultat_%s_%s.json" % (etape, nom), "w", encoding="utf-8") as f:
        json.dump(out, f, indent=2, ensure_ascii=False)

log2 = dict(seed=SEED, device=dev, n={s: len(df[s]) for s in S}, reseau_meilleur_validation=best,
            gains_vs_resnet18=gains, res=res2, table=tab2.to_dict(), per_type=per_type2.to_dict())
with open(OUT + "results/e3_autres_reseaux_metrics.json", "w", encoding="utf-8") as f:
    json.dump(log2, f, indent=2, ensure_ascii=False,
              default=lambda o: o.item() if hasattr(o, "item") else str(o))

for root, _, files in os.walk(OUT):
    for fn in sorted(files):
        p = os.path.join(root, fn)
        print(p, round(os.path.getsize(p) / 1e6, 2), "Mo")

/kaggle/working/__notebook__.ipynb 0.06 Mo
/kaggle/working/results/e2_e3_metrics.json 0.0 Mo
/kaggle/working/results/e2_test_proba.npy 0.01 Mo
/kaggle/working/results/e2_val_proba.npy 0.01 Mo
/kaggle/working/results/e3_autres_reseaux_metrics.json 0.0 Mo
/kaggle/working/results/e3_test_proba.npy 0.01 Mo
/kaggle/working/results/e3_val_proba.npy 0.01 Mo
/kaggle/working/results/e3b_test_proba.npy 0.01 Mo
/kaggle/working/results/e3b_val_proba.npy 0.01 Mo
/kaggle/working/results/e3c_test_proba.npy 0.01 Mo
/kaggle/working/results/e3c_val_proba.npy 0.01 Mo
/kaggle/working/results/e3ctrl_test_proba.npy 0.01 Mo
/kaggle/working/results/e3ctrl_val_proba.npy 0.01 Mo
/kaggle/working/results/e3d_test_proba.npy 0.01 Mo
/kaggle/working/results/e3d_val_proba.npy 0.01 Mo
/kaggle/working/results/mlflow/resultat_E3b_efficientnet_b0.json 0.0 Mo
/kaggle/working/results/mlflow/resultat_E3c_vit_b_16.json 0.0 Mo
/kaggle/working/results/mlflow/resultat_E3ctrl_resnet18.json 0.0 Mo
/kaggle/working/results/mlflow/r